In [2]:
import pandas as pd

# 1. สร้าง DataFrame จากข้อมูลของคุณ
# data = 'data/data.csv'

# column_names = ["AnimalName", "symptoms1", "symptoms2", "symptoms3", "symptoms4", "symptoms5", "Dangerous"]
# df = pd.DataFrame(data, columns=column_names)

df = pd.read_csv('data/data.csv')

# 2. หาจำนวนชนิดสัตว์ทั้งหมด (Unique Animals)
unique_animals = df['AnimalName'].unique()
print(f"--- จำนวนชนิดสัตว์ทั้งหมด: {len(unique_animals)} ชนิด ---")
print(unique_animals)

# 3. หาอาการทั้งหมดที่มีใน Dataset (Unique Symptoms)
# รวมทุกคอลัมน์ symptoms เข้าด้วยกันแล้วหาค่าที่ไม่ซ้ำ
all_symptoms = pd.unique(df[['symptoms1', 'symptoms2', 'symptoms3', 'symptoms4', 'symptoms5']].values.ravel())
# กรองค่าที่เป็น None หรือค่าว่างออก (ถ้ามี)
all_symptoms = [s for s in all_symptoms if s and str(s) != 'nan']

print(f"\n--- จำนวนอาการทั้งหมดที่พบ: {len(all_symptoms)} อาการ ---")
print(sorted(all_symptoms))

# 4. (แถม) แยกอาการตามชนิดสัตว์
print("\n--- ตัวอย่าง: จำนวนอาการแยกตามชนิดสัตว์ ---")
animal_symptoms_count = df.groupby('AnimalName').size()
print(animal_symptoms_count)

--- จำนวนชนิดสัตว์ทั้งหมด: 46 ชนิด ---
['Dog' 'cat' 'Rabbit' 'cow' 'chicken' 'cattle' 'mammal' 'Cattle' 'Horse'
 'Turtle' 'Hamster' 'Lion' 'Fox' 'Fox ' 'Goat' 'Deer' 'Chicken' 'Monkey'
 'Birds' 'Sheep' 'Pigs' 'Fowl' 'Duck' 'Other Birds' 'snake' 'horse' 'duck'
 'donkey' 'Donkey' 'mules' 'Elephant' 'Elk' 'Wapiti' 'Mule deer'
 'Black-tailed deer' 'Sika deer' 'White-tailed deer' 'Reindeer' 'Moos'
 'Tiger' 'Goats' 'Buffaloes' 'Dogs' 'Wolves' 'Hyaenas' 'Pig']

--- จำนวนอาการทั้งหมดที่พบ: 935 อาการ ---
[' abortion at the end of gestation', ' dizziness and fainting', 'Abdminal pain', 'Abdominal destention', 'Abdominal discomfort', 'Abdominal pain', 'Abdonormal discomfort', 'Abdonormal pain', 'Abnormal behaviour', 'Abnormal conformation', 'Abnormalities', 'Abnormally long leg', 'Abortion', 'Abortion ', 'Abortion on late pregancy', 'Achomotrica', 'Acting aggressive', 'Acting unnaturally tame', 'Agalactia', 'Aggressiveness', 'Air sacculitis', 'Allergic Reaction', 'Anaemia', 'Aneamia', 'Anemia', '

In [4]:
import pandas as pd
import os

# 1. จัดเตรียมข้อมูล (สมมติว่า data_list คือข้อมูลดิบของคุณ)
# ในที่นี้ผมจำลองโครงสร้างจากที่คุณส่งมา
column_names = ["AnimalName", "symptoms1", "symptoms2", "symptoms3", "symptoms4", "symptoms5", "Dangerous"]
# ... (สมมติว่าอ่านไฟล์จาก csv เดิมของคุณมาเป็น df) ...
# df = pd.read_csv('your_raw_data.csv') 

def clean_and_save_data(df):
    # สร้างโฟลเดอร์ data ถ้ายังไม่มี
    if not os.path.exists('data'):
        os.makedirs('data')

    # --- 2. Clean Animal Names ---
    # ลบช่องว่าง และทำให้เป็นตัวพิมพ์เล็กก่อนเพื่อเช็กความซ้ำ
    df['AnimalName'] = df['AnimalName'].str.strip().str.capitalize()
    
    # Mapping เพื่อรวมกลุ่ม (ตัวอย่างการยุบกลุ่มที่ซ้ำซ้อน)
    animal_map = {
        'Dogs': 'Dog',
        'Cat': 'Cat', # ในกรณีที่มี cat/Cat
        'Cattle': 'Cattle',
        'Cow': 'Cattle',
        'Buffaloes': 'Cattle',
        'Chicken': 'Poultry',
        'Chicken ': 'Poultry',
        'Birds': 'Poultry',
        'Duck': 'Poultry',
        'Fowl': 'Poultry',
        'Goats': 'Goat',
        'Pigs': 'Pig',
        'Donkey': 'Donkey',
        'Mules': 'Donkey',
        'Horse': 'Horse',
        'Tiger': 'Wild_Cat',
        'Lion': 'Wild_Cat'
    }
    df['AnimalName'] = df['AnimalName'].replace(animal_map)

    # --- 3. Clean Symptoms ---
    symptom_cols = ['symptoms1', 'symptoms2', 'symptoms3', 'symptoms4', 'symptoms5']
    
    for col in symptom_cols:
        # ลบช่องว่าง และทำเป็นตัวพิมพ์เล็กทั้งหมดเพื่อลดความซ้ำซ้อนของคำ
        df[col] = df[col].str.strip().str.lower()
        
        # แก้ไขคำสะกดผิดที่พบบ่อย (ตัวอย่าง)
        df[col] = df[col].replace({
            'pnemonia': 'pneumonia',
            'weekness': 'weakness',
            'diarrhoea': 'diarrhea',
            'seizuers': 'seizures',
            'anoxeria': 'anorexia'
        })

    # --- 4. ลบข้อมูลที่ซ้ำกันจริงๆ (Duplicate Rows) ---
    df = df.drop_duplicates()

    # --- 5. บันทึกไฟล์ ---
    file_path = 'data/cleaned_animal_data.csv'
    df.to_csv(file_path, index=False, encoding='utf-8-sig')
    
    print(f"✅ Clean Data สำเร็จ!")
    print(f"📍 ไฟล์ถูกเก็บไว้ที่: {file_path}")
    print(f"📊 จำนวนชนิดสัตว์หลัง Clean: {df['AnimalName'].nunique()}")
    return df

# เรียกใช้งานฟังก์ชัน
cleaned_df = clean_and_save_data(df)

✅ Clean Data สำเร็จ!
📍 ไฟล์ถูกเก็บไว้ที่: data/cleaned_animal_data.csv
📊 จำนวนชนิดสัตว์หลัง Clean: 30


In [11]:
df = pd.read_csv('data/data.csv')
df

,AnimalName,symptoms1,symptoms2,symptoms3,symptoms4,symptoms5,Dangerous
0,Dog,Fever,Diarrhea,Vomiting,Weight loss,Dehydration,Yes
1,Dog,Fever,Diarrhea,Coughing,Tiredness,Pains,Yes
2,Dog,Fever,Diarrhea,Coughing,Vomiting,Anorexia,Yes
3,Dog,Fever,Difficulty breathing,Coughing,Lethargy,Sneezing,Yes
4,Dog,Fever,Diarrhea,Coughing,Lethargy,Blue Eye,Yes
...,...,...,...,...,...,...,...
866,Buffaloes,Fever,Difficulty breathing,Poor Appetite,Eye and Skin change,Unable to exercise,Yes
867,Buffaloes,Fever,Loss of appetite,Lession on the skin,Lethargy,Joint Pain,Yes
868,Buffaloes,Lesions in the nasal cavity,Lesions on nose,Vomiting,Noisy Breathing,Lesions on nose,Yes
869,Buffaloes,Hair loss,Dandruff,Vomiting,Crusting of the skin,Ulcerated skin,Yes


In [ ]:
import pandas as pd

# โหลดข้อมูลที่คลีนแล้ว
df = pd.read_csv('data/cleaned_animal_data.csv')

# 1. ปรับโครงสร้างข้อมูลอาการจาก 5 คอลัมน์ให้เป็น List เดียว
symptom_cols = ['symptoms1', 'symptoms2', 'symptoms3', 'symptoms4', 'symptoms5']
all_symptoms_series = df[symptom_cols].stack()

# 2. นับความถี่ของแต่ละอาการ (Symptom Frequency)
symptom_counts = all_symptoms_series.value_counts()

# 3. เช็คความผันผวนโดยดูว่าอาการนั้นปรากฏในสัตว์ "กี่ชนิด" (Species Diversity)
# ยิ่งเจอในสัตว์หลายชนิด แปลว่าความผันผวนเชิงสถิติจะสูง (Common Symptoms)
symptom_diversity = {}
for symptom in symptom_counts.index:
    # หาว่าอาการนี้พบในสัตว์กี่ชนิด
    species_count = df[df[symptom_cols].apply(lambda row: symptom in row.values, axis=1)]['AnimalName'].nunique()
    symptom_diversity[symptom] = species_count

# 4. รวมข้อมูลเข้าด้วยกัน
volatility_df = pd.DataFrame({
    'Frequency': symptom_counts,
    'Species_Spread': pd.Series(symptom_diversity)
}).sort_values(by=['Frequency', 'Species_Spread'], ascending=False)

print("--- 10 อันดับอาการที่มีความผันผวน/ความถี่สูงที่สุด ---")
print(volatility_df.head(10))

--- 10 อันดับอาการที่มีความผันผวน/ความถี่สูงที่สุด ---
                  Frequency  Species_Spread
fever                   275              17
weight loss             166              17
diarrhea                156              18
coughing                110              14
loss of appetite        102              16
depression               86              17
lethargy                 83              14
pains                    83              12
vomiting                 75              14
pain                     74              15


In [22]:
all_symptoms_series.value_counts()

fever                 275
weight loss           166
diarrhea              156
coughing              110
loss of appetite      102
                     ... 
head tossing            1
bleeding wounds         1
thivk skin              1
dizzines                1
anversion to light      1
Name: count, Length: 864, dtype: int64

In [28]:
import pandas as pd

# โหลดข้อมูล
df = pd.read_csv('data/cleaned_animal_data.csv')

# 1. กรองเฉพาะแถวที่ Dangerous = 'Yes' (ซึ่งในเคสนี้คือทั้งหมด)
yes_df = df[df['Dangerous'] == 'Yes']

# 2. รวมอาการจากทุกคอลัมน์ (symptoms1-5) เข้ามานับรวมกัน
symptom_cols = ['symptoms1', 'symptoms2', 'symptoms3', 'symptoms4', 'symptoms5']
symptom_series = yes_df[symptom_cols].stack().str.strip().str.lower()

# 3. นับความถี่ของแต่ละอาการ
symptom_counts = symptom_series.value_counts().reset_index()
symptom_counts.columns = ['Symptom', 'Count_in_Dangerous_Cases']

# 4. คำนวณเป็นเปอร์เซ็นต์ (ความน่าจะเป็นที่อาการนี้จะอยู่ในเคส Yes)
total_cases = len(yes_df)
symptom_counts['Percentage'] = (symptom_counts['Count_in_Dangerous_Cases'] / total_cases * 100).round(2)

print(f"--- Top 10 อาการที่พบในเคสอันตราย (Yes) บ่อยที่สุด ---")
print(symptom_counts.head(10))

--- Top 10 อาการที่พบในเคสอันตราย (Yes) บ่อยที่สุด ---
            Symptom  Count_in_Dangerous_Cases  Percentage
0             fever                       273       33.33
1       weight loss                       160       19.54
2          diarrhea                       146       17.83
3          coughing                       106       12.94
4  loss of appetite                        98       11.97
5             pains                        83       10.13
6          lethargy                        83       10.13
7        depression                        82       10.01
8          vomiting                        75        9.16
9              pain                        70        8.55


# Preprocessing

In [30]:
import pandas as pd
import numpy as np

def preprocess_animal_data(df):
    # --- 1. จัดกลุ่มสัตว์ (Animal Grouping) ---
    # ทำให้เป็นมาตรฐาน (ลบช่องว่าง, ตัวพิมพ์ใหญ่)
    df['AnimalName'] = df['AnimalName'].str.strip().str.capitalize()
    
    animal_mapping = {
        'Dogs': 'Dog', 'Cat': 'Cat', 'Cattle': 'Cattle', 'Cow': 'Cattle',
        'Buffaloes': 'Cattle', 'Chicken': 'Poultry', 'Birds': 'Poultry',
        'Duck': 'Poultry', 'Fowl': 'Poultry', 'Goats': 'Goat',
        'Pigs': 'Pig', 'Horse': 'Horse', 'Lion': 'Wild_Cat', 'Tiger': 'Wild_Cat'
    }
    df['Animal_Group'] = df['AnimalName'].replace(animal_mapping)
    # สัตว์ที่ไม่อยู่ในกลุ่มหลัก ให้เป็น 'Other'
    main_groups = ['Dog', 'Cat', 'Cattle', 'Poultry', 'Goat', 'Pig', 'Horse', 'Wild_Cat']
    df.loc[~df['Animal_Group'].isin(main_groups), 'Animal_Group'] = 'Other'

    # --- 2. ทำความสะอาดอาการ (Symptom Cleaning) ---
    symptom_cols = ['symptoms1', 'symptoms2', 'symptoms3', 'symptoms4', 'symptoms5']
    
    # ยุบรวมอาการที่ความหมายเดียวกัน
    symptom_mapping = {
        'pains': 'pain', 'vomitting': 'vomiting', 'diarrhoea': 'diarrhea',
        'pnemonia': 'pneumonia', 'weekness': 'weakness', 'seizuers': 'seizures',
        'anoxeria': 'anorexia', 'loss of  appetite': 'loss of appetite'
    }

    def clean_symptom(s):
        if pd.isna(s): return None
        s = str(s).strip().lower()
        return symptom_mapping.get(s, s)

    for col in symptom_cols:
        df[col] = df[col].apply(clean_symptom)

    # --- 3. ทำ One-Hot Encoding (แปลงเป็นตาราง 0 และ 1) ---
    # ดึงอาการทั้งหมดมาเป็น List (เฉพาะที่สำคัญ เช่น เจอมากกว่า 2 ครั้ง)
    all_symptoms = df[symptom_cols].stack().value_counts()
    common_symptoms = all_symptoms[all_symptoms >= 2].index.tolist()

    # สร้างคอลัมน์ใหม่สำหรับแต่ละอาการ
    for symptom in common_symptoms:
        df[f'symptom_{symptom}'] = df[symptom_cols].apply(lambda x: 1 if symptom in x.values else 0, axis=1)

    # แปลง Animal_Group เป็น One-Hot ด้วย
    df = pd.get_dummies(df, columns=['Animal_Group'], prefix='group')

    return df, common_symptoms

# เรียกใช้งาน
processed_df, symptom_list = preprocess_animal_data(df)

C:\Users\user\AppData\Local\Temp\ipykernel_34828\2976895630.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'symptom_{symptom}'] = df[symptom_cols].apply(lambda x: 1 if symptom in x.values else 0, axis=1)
C:\Users\user\AppData\Local\Temp\ipykernel_34828\2976895630.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'symptom_{symptom}'] = df[symptom_cols].apply(lambda x: 1 if symptom in x.values else 0, axis=1)
C:\Users\user\AppData\Local\Temp\ipykernel_34828\2976895630.py:45: PerformanceWarning: DataFrame is highly

In [31]:
import pandas as pd
import json
import os
from googletrans import Translator
import time

def full_preprocess_pipeline(df):
    # 1. สร้าง Folder สำหรับเก็บข้อมูล
    if not os.path.exists('data'):
        os.makedirs('data')
    
    translator = Translator()
    
    # --- 2. Clean Animal Names ---
    df['AnimalName'] = df['AnimalName'].str.strip().str.capitalize()
    
    # --- 3. Clean & Collect All Symptoms ---
    symptom_cols = ['symptoms1', 'symptoms2', 'symptoms3', 'symptoms4', 'symptoms5']
    all_symptoms_raw = df[symptom_cols].stack().str.strip().str.lower().unique()
    all_symptoms_raw = [s for s in all_symptoms_raw if s and s != 'nan']

    # --- 4. แปลภาษาอัตโนมัติด้วย API (Auto-Translation) ---
    print(f"กำลังแปล {len(all_symptoms_raw)} อาการเป็นภาษาไทย...")
    symptom_dict = {}
    
    # แบ่งแปลทีละนิดเพื่อป้องกัน API Block
    for s in all_symptoms_raw:
        try:
            # แปลจากอังกฤษเป็นไทย
            translation = translator.translate(s, src='en', dest='th').text
            symptom_dict[s] = translation
            print(f" แปลแล้ว: {s} -> {translation}")
            time.sleep(0.2) # หน่วงเวลาเล็กน้อย
        except Exception as e:
            symptom_dict[s] = s # ถ้าแปลไม่ได้ให้ใช้คำเดิม
            print(f" ! แปลไม่สำเร็จ: {s}")

    # บันทึกพจนานุกรมภาษาไทย
    with open('data/symptom_dictionary.json', 'w', encoding='utf-8') as f:
        json.dump(symptom_dict, f, ensure_ascii=False, indent=4)

    # --- 5. สร้าง cleaned_features.csv (One-Hot Matrix) ---
    feature_df = pd.DataFrame(index=df.index)
    feature_df['AnimalName'] = df['AnimalName']
    
    for symptom in all_symptoms_raw:
        # เช็คว่าแถวนั้นมีอาการนี้หรือไม่ (1 หรือ 0)
        feature_df[symptom] = df[symptom_cols].apply(
            lambda x: 1 if symptom in [str(i).strip().lower() for i in x.values] else 0, 
            axis=1
        )
    
    feature_df['Target_Dangerous'] = df['Dangerous'].map({'Yes': 1, 'No': 0})
    feature_df.to_csv('data/cleaned_features.csv', index=False, encoding='utf-8-sig')

    print("\n--- ✅ สรุปการทำงาน ---")
    print(f"1. ไฟล์ตาราง 0/1: data/cleaned_features.csv")
    print(f"2. ไฟล์พจนานุกรมไทย: data/symptom_dictionary.json")
    
    return feature_df, symptom_dict

# รัน Pipeline
processed_df, thai_dict = full_preprocess_pipeline(df)

กำลังแปล 861 อาการเป็นภาษาไทย...
 แปลแล้ว: fever -> ไข้
 แปลแล้ว: diarrhea -> ท้องเสีย
 แปลแล้ว: vomiting -> อาเจียน
 แปลแล้ว: weight loss -> ลดน้ำหนัก
 แปลแล้ว: dehydration -> การคายน้ำ
 แปลแล้ว: coughing -> ไอ
 แปลแล้ว: tiredness -> ความเหนื่อยล้า
 แปลแล้ว: pain -> ความเจ็บปวด
 แปลแล้ว: anorexia -> อาการเบื่ออาหาร
 แปลแล้ว: difficulty breathing -> หายใจลำบาก
 แปลแล้ว: lethargy -> ความง่วง
 แปลแล้ว: sneezing -> จาม
 แปลแล้ว: blue eye -> ตาสีฟ้า
 แปลแล้ว: respiratory distress -> ความทุกข์ทางเดินหายใจ
 แปลแล้ว: seizures -> อาการชัก
 แปลแล้ว: hyperesthesia -> ความรู้สึกเกินปกติ
 แปลแล้ว: sudden death -> เสียชีวิตอย่างกะทันหัน
 แปลแล้ว: ulcers -> แผลพุพอง
 แปลแล้ว: poor appetite -> ความอยากอาหารไม่ดี
 แปลแล้ว: tarry stool -> อุจจาระรออยู่
 แปลแล้ว: enlarged lymph nodes -> ต่อมน้ำเหลืองโต
 แปลแล้ว: facial swelling -> ใบหน้าบวม
 แปลแล้ว: bloody drool -> น้ำลายไหลเปื้อนเลือด
 แปลแล้ว: foul breath -> ลมหายใจเหม็น
 แปลแล้ว: unable to eat -> ไม่สามารถกินได้
 แปลแล้ว: lossened teeth -> ฟันที่สูญ

C:\Users\user\AppData\Local\Temp\ipykernel_34828\3475641021.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_df[symptom] = df[symptom_cols].apply(
C:\Users\user\AppData\Local\Temp\ipykernel_34828\3475641021.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_df[symptom] = df[symptom_cols].apply(
C:\Users\user\AppData\Local\Temp\ipykernel_34828\3475641021.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Co


--- ✅ สรุปการทำงาน ---
1. ไฟล์ตาราง 0/1: data/cleaned_features.csv
2. ไฟล์พจนานุกรมไทย: data/symptom_dictionary.json


C:\Users\user\AppData\Local\Temp\ipykernel_34828\3475641021.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_df[symptom] = df[symptom_cols].apply(
C:\Users\user\AppData\Local\Temp\ipykernel_34828\3475641021.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_df[symptom] = df[symptom_cols].apply(
C:\Users\user\AppData\Local\Temp\ipykernel_34828\3475641021.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Co

In [12]:
import pandas as pd
import os

def clean_and_save_features():
    file_path = 'data/cleaned_features.csv'
    
    if not os.path.exists(file_path):
        print(f"❌ ไม่พบไฟล์ที่: {file_path}")
        return

    # 1. โหลดข้อมูล
    df = pd.read_csv(file_path)
    initial_count = len(df)
    print(f"📊 จำนวนข้อมูลเริ่มต้น: {initial_count} แถว")

    # 2. ลบแถวที่ Target_Dangerous เป็นค่าว่าง (NaN)
    # สาเหตุส่วนใหญ่เกิดจากมีแถวว่างท้ายไฟล์ CSV หรือแมปค่า Yes/No ไม่ติด
    df = df.dropna(subset=['Target_Dangerous'])
    
    # 3. เติมค่า 0 ในส่วนที่เป็น Features (อาการ) ที่เป็นค่าว่าง
    # เพื่อป้องกัน Error ตอนเทรนโมเดล
    df = df.fillna(0)

    # 4. ตรวจสอบประเภทข้อมูลของ Target (ควรเป็น int)
    df['Target_Dangerous'] = df['Target_Dangerous'].astype(int)

    final_count = len(df)
    
    # 5. บันทึกไฟล์ทับที่เดิม (หรือจะเปลี่ยนชื่อไฟล์ก็ได้)
    df.to_csv(file_path, index=False, encoding='utf-8-sig')
    
    print(f"✅ คลีนข้อมูลเสร็จเรียบร้อย!")
    print(f"🗑️ ลบไปทั้งหมด: {initial_count - final_count} แถว")
    print(f"💾 บันทึกไฟล์ทับที่: {file_path} (คงเหลือ {final_count} แถว)")
    print("-" * 30)
    print("ค่าใน Target ตอนนี้:")
    print(df['Target_Dangerous'].value_counts())

# รันฟังก์ชันเพื่อคลีนไฟล์
clean_and_save_features()

📊 จำนวนข้อมูลเริ่มต้น: 839 แถว
✅ คลีนข้อมูลเสร็จเรียบร้อย!
🗑️ ลบไปทั้งหมด: 0 แถว
💾 บันทึกไฟล์ทับที่: data/cleaned_features.csv (คงเหลือ 839 แถว)
------------------------------
ค่าใน Target ตอนนี้:
Target_Dangerous
1    819
0     20
Name: count, dtype: int64


In [15]:
import pandas as pd
from imblearn.over_sampling import SVMSMOTE
from sklearn.model_selection import train_test_split

# โหลดข้อมูลขึ้นมาก่อน (ตรวจสอบให้แน่ใจว่าไฟล์ cleaned_features.csv อยู่ในโฟลเดอร์ data)
feature_df = pd.read_csv('data/cleaned_features.csv') 

# 1. เตรียมข้อมูล X และ y
# ตัดคอลัมน์ที่ไม่ใช่ Feature ออก (เช่น ชื่อสัตว์) และแยก Target ออกมา
X = feature_df.drop(columns=['AnimalName', 'Target_Dangerous'])
y = feature_df['Target_Dangerous']

# เติมค่าว่าง (ถ้ามี) เป็น 0 ก่อนทำ SMOTE
X = X.fillna(0)

print("--- Category value distribution before SVMSMOTE ---")
print(y.value_counts())

# 2. เริ่มทำ SVMSMOTE
svm_smote = SVMSMOTE(sampling_strategy='auto', random_state=42)

try:
    X_resampled, y_resampled = svm_smote.fit_resample(X, y)
    
    print("\n--- Category value distribution after SVMSMOTE ---")
    print(pd.Series(y_resampled).value_counts())
    
    # 3. เก็บข้อมูลที่ทำ Balance แล้วลง DataFrame ใหม่
    balanced_df = pd.concat([pd.DataFrame(X_resampled), pd.Series(y_resampled, name='Target_Dangerous')], axis=1)
    
    # 4. บันทึกเป็นไฟล์ CSV ตามชื่อที่ต้องการ
    balanced_df.to_csv('data/cleaned_features_balance.csv', index=False)
    print("\n✅ บันทึกไฟล์สำเร็จ! ชื่อไฟล์: data/cleaned_features_balance.csv")
    
except ValueError as e:
    print(f"\n❌ เกิดข้อผิดพลาด: {e}")
    print("คำแนะนำ: SVMSMOTE ต้องการข้อมูลคลาสตัวน้อย (Minority Class) อย่างน้อย 6 ตัวอย่างขึ้นไป")

--- Category value distribution before SVMSMOTE ---
Target_Dangerous
1    819
0     20
Name: count, dtype: int64

--- Category value distribution after SVMSMOTE ---
Target_Dangerous
1    819
0    459
Name: count, dtype: int64

✅ บันทึกไฟล์สำเร็จ! ชื่อไฟล์: data/cleaned_features_balance.csv
